# Summary Statistics and Reporting

This notebook demonstrates comprehensive techniques for generating summary statistics and creating statistical reports in data science projects. We'll cover both basic descriptive statistics and more advanced reporting methods.

## 1. Import Essential Libraries

First, let's import all the necessary libraries for statistical analysis and reporting.

In [ ]:
# Import core data science libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import statistical libraries
from scipy import stats

# Import automated reporting libraries
import warnings
warnings.filterwarnings('ignore')

# Set visualization styles
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

# Display settings for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

## 2. Load and Prepare Sample Data

We'll use multiple datasets to demonstrate different statistical reporting techniques:
1. A built-in dataset from seaborn (tips)
2. A synthetic dataset with various distributions

In [ ]:
# Load the tips dataset from seaborn
tips_df = sns.load_dataset('tips')

# Create a synthetic dataset with different distributions
np.random.seed(42)
n_samples = 1000

# Create a DataFrame with different distributions
synthetic_df = pd.DataFrame({
    'normal': np.random.normal(loc=50, scale=10, size=n_samples),
    'uniform': np.random.uniform(low=0, high=100, size=n_samples),
    'exponential': np.random.exponential(scale=10, size=n_samples),
    'bimodal': np.concatenate([
        np.random.normal(loc=30, scale=5, size=n_samples//2),
        np.random.normal(loc=70, scale=5, size=n_samples//2)
    ]),
    'categorical': np.random.choice(['A', 'B', 'C', 'D'], size=n_samples, p=[0.4, 0.3, 0.2, 0.1])
})

# Add some missing values to demonstrate handling
for col in synthetic_df.columns[:-1]:  # Skip categorical column
    mask = np.random.random(n_samples) < 0.05  # 5% missing values
    synthetic_df.loc[mask, col] = np.nan

# Display the first rows of each dataset
print("Tips Dataset:")
display(tips_df.head())

print("\nSynthetic Dataset:")
display(synthetic_df.head())

### Data Preparation

Let's handle missing values and check data types before proceeding with analysis.

In [ ]:
# Check for missing values in both datasets
print("Missing values in Tips Dataset:")
print(tips_df.isnull().sum())

print("\nMissing values in Synthetic Dataset:")
print(synthetic_df.isnull().sum())

# Handle missing values in the synthetic dataset
synthetic_df_clean = synthetic_df.copy()
for col in synthetic_df.columns[:-1]:  # Skip categorical column
    # Impute missing values with median for numeric columns
    synthetic_df_clean[col].fillna(synthetic_df[col].median(), inplace=True)

# Verify data types
print("\nData Types - Tips Dataset:")
print(tips_df.dtypes)

print("\nData Types - Synthetic Dataset:")
print(synthetic_df_clean.dtypes)

## 3. Calculate Descriptive Statistics

Now let's calculate basic descriptive statistics for our datasets including:
- Measures of central tendency (mean, median, mode)
- Measures of dispersion (standard deviation, variance, range)
- Measures of shape (skewness, kurtosis)

In [ ]:
def calculate_descriptive_stats(df, numeric_only=True):
    """Calculate comprehensive descriptive statistics for a dataframe"""
    if numeric_only:
        df_numeric = df.select_dtypes(include=[np.number])
    else:
        df_numeric = df
    
    # Basic statistics from pandas
    basic_stats = df_numeric.describe()
    
    # Additional statistics
    additional_stats = pd.DataFrame(index=df_numeric.columns)
    additional_stats.loc['range'] = df_numeric.max() - df_numeric.min()
    additional_stats.loc['variance'] = df_numeric.var()
    additional_stats.loc['skewness'] = df_numeric.skew()
    additional_stats.loc['kurtosis'] = df_numeric.kurt()
    additional_stats.loc['median'] = df_numeric.median()
    additional_stats.loc['mode'] = df_numeric.mode().iloc[0] if not df_numeric.empty else None
    additional_stats.loc['missing'] = df_numeric.isnull().sum()
    additional_stats.loc['missing_pct'] = (df_numeric.isnull().sum() / len(df_numeric)) * 100
    
    # Combine statistics
    all_stats = pd.concat([basic_stats, additional_stats.loc[~additional_stats.index.isin(basic_stats.index)]])
    
    return all_stats

# Calculate statistics for Tips dataset
tips_stats = calculate_descriptive_stats(tips_df)
print("Descriptive Statistics for Tips Dataset:")
display(tips_stats)

# Calculate statistics for Synthetic dataset
synthetic_stats = calculate_descriptive_stats(synthetic_df_clean)
print("\nDescriptive Statistics for Synthetic Dataset:")
display(synthetic_stats)

### Confidence Intervals

Let's calculate confidence intervals for the mean of each numeric variable to understand the precision of our estimates.

In [ ]:
def calculate_confidence_intervals(df, confidence=0.95):
    """Calculate confidence intervals for the mean of each numeric column"""
    df_numeric = df.select_dtypes(include=[np.number])
    
    result = pd.DataFrame(index=df_numeric.columns)
    result['mean'] = df_numeric.mean()
    
    for column in df_numeric.columns:
        data = df_numeric[column].dropna()
        n = len(data)
        mean = data.mean()
        std_err = stats.sem(data)
        h = std_err * stats.t.ppf((1 + confidence) / 2, n - 1)
        
        result.loc[column, 'lower_ci'] = mean - h
        result.loc[column, 'upper_ci'] = mean + h
        result.loc[column, 'ci_width'] = h * 2
    
    return result

# Calculate 95% confidence intervals for Tips dataset
tips_ci = calculate_confidence_intervals(tips_df)
print("95% Confidence Intervals for Tips Dataset:")
display(tips_ci)

# Calculate 95% confidence intervals for Synthetic dataset
synthetic_ci = calculate_confidence_intervals(synthetic_df_clean)
print("\n95% Confidence Intervals for Synthetic Dataset:")
display(synthetic_ci)

## 4. Generate Statistical Summaries

Let's use pandas built-in functions to create comprehensive summaries, including groupby operations for segmented analysis.

In [ ]:
# Basic info about the Tips dataset
print("Tips Dataset Info:")
tips_df.info()

# Value counts for categorical variables in Tips dataset
print("\nTips Dataset - Day Distribution:")
print(tips_df['day'].value_counts())

print("\nTips Dataset - Time Distribution:")
print(tips_df['time'].value_counts())

# Groupby operations for segmented analysis
print("\nTips by Day and Time:")
display(tips_df.groupby(['day', 'time']).agg({
    'total_bill': ['count', 'mean', 'std', 'min', 'max'],
    'tip': ['mean', 'std', 'min', 'max'],
    'tip_pct': lambda x: (tips_df.loc[x.index, 'tip'] / tips_df.loc[x.index, 'total_bill']).mean()
}).round(2))

# Let's add a tip percentage column for analysis
tips_df['tip_pct'] = tips_df['tip'] / tips_df['total_bill']

# Group statistics by smoker/non-smoker
print("\nTips Statistics by Smoker Status:")
display(tips_df.groupby('smoker').agg({
    'total_bill': ['count', 'mean', 'median', 'std'],
    'tip': ['mean', 'median', 'std'],
    'tip_pct': ['mean', 'median', 'std']
}).round(3))

# For synthetic data, let's analyze numeric columns
print("\nSynthetic Dataset - Basic Statistics:")
display(synthetic_df_clean.describe().T)

# Distribution of categorical variable
print("\nSynthetic Dataset - Categorical Distribution:")
display(synthetic_df_clean['categorical'].value_counts(normalize=True).to_frame('percentage').reset_index().rename(columns={'index': 'category'}))

## 5. Visual Statistical Reporting

Now let's create visualizations to illustrate the statistical properties of our data.

In [ ]:
# Create a function for visualizing distributions
def visualize_distributions(df, numeric_columns, categorical_columns=None):
    """Visualize distributions of numeric and categorical columns"""
    
    # Numeric distributions
    if len(numeric_columns) > 0:
        fig, axes = plt.subplots(len(numeric_columns), 3, figsize=(18, 5 * len(numeric_columns)))
        
        if len(numeric_columns) == 1:
            axes = axes.reshape(1, -1)
            
        for i, col in enumerate(numeric_columns):
            # Histogram
            sns.histplot(df[col].dropna(), kde=True, ax=axes[i, 0])
            axes[i, 0].set_title(f'Distribution of {col}')
            axes[i, 0].set_xlabel(col)
            
            # Box plot
            sns.boxplot(y=df[col].dropna(), ax=axes[i, 1])
            axes[i, 1].set_title(f'Box Plot of {col}')
            axes[i, 1].set_ylabel(col)
            
            # QQ plot to check normality
            stats.probplot(df[col].dropna(), plot=axes[i, 2])
            axes[i, 2].set_title(f'QQ Plot of {col}')
        
        plt.tight_layout()
        plt.show()
    
    # Categorical distributions
    if categorical_columns and len(categorical_columns) > 0:
        fig, axes = plt.subplots(1, len(categorical_columns), figsize=(6 * len(categorical_columns), 5))
        
        if len(categorical_columns) == 1:
            axes = [axes]
            
        for i, col in enumerate(categorical_columns):
            value_counts = df[col].value_counts().sort_values(ascending=False)
            sns.barplot(x=value_counts.index, y=value_counts.values, ax=axes[i])
            axes[i].set_title(f'Distribution of {col}')
            axes[i].set_ylabel('Count')
            axes[i].tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()

# Visualize distributions for tips dataset
print("Distributions of Numeric Variables in Tips Dataset:")
visualize_distributions(tips_df, ['total_bill', 'tip', 'tip_pct'], ['day', 'time', 'sex', 'smoker'])

# Visualize distributions for synthetic dataset
print("\nDistributions of Numeric Variables in Synthetic Dataset:")
visualize_distributions(synthetic_df_clean, ['normal', 'uniform', 'exponential', 'bimodal'], ['categorical'])

In [ ]:
# Create correlation matrix visualization for numeric variables
def visualize_correlations(df, title="Correlation Matrix"):
    """Create a correlation matrix visualization"""
    # Select only numeric columns
    numeric_df = df.select_dtypes(include=[np.number])
    
    # Calculate correlation matrix
    corr_matrix = numeric_df.corr()
    
    # Create heatmap
    plt.figure(figsize=(10, 8))
    mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
    sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", 
                mask=mask, vmin=-1, vmax=1, linewidths=0.5)
    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.show()
    
    return corr_matrix

# Visualize correlations for tips dataset
tips_corr = visualize_correlations(tips_df, "Correlation Matrix - Tips Dataset")

# Visualize correlations for synthetic dataset
synthetic_corr = visualize_correlations(synthetic_df_clean, "Correlation Matrix - Synthetic Dataset")

In [ ]:
# Create plots for segmented analysis
plt.figure(figsize=(14, 10))

# Create a 2x2 grid of plots
plt.subplot(2, 2, 1)
sns.boxplot(x="day", y="total_bill", data=tips_df)
plt.title("Total Bill by Day", fontsize=14)

plt.subplot(2, 2, 2)
sns.boxplot(x="day", y="tip", data=tips_df)
plt.title("Tip by Day", fontsize=14)

plt.subplot(2, 2, 3)
sns.boxplot(x="time", y="total_bill", data=tips_df)
plt.title("Total Bill by Time", fontsize=14)

plt.subplot(2, 2, 4)
sns.boxplot(x="time", y="tip", data=tips_df)
plt.title("Tip by Time", fontsize=14)

plt.tight_layout()
plt.show()

# Interaction between categorical variables and tip percentage
plt.figure(figsize=(14, 6))

plt.subplot(1, 2, 1)
sns.barplot(x="day", y="tip_pct", hue="sex", data=tips_df)
plt.title("Tip Percentage by Day and Gender", fontsize=14)
plt.ylim(0, 0.25)

plt.subplot(1, 2, 2)
sns.barplot(x="time", y="tip_pct", hue="smoker", data=tips_df)
plt.title("Tip Percentage by Time and Smoker Status", fontsize=14)
plt.ylim(0, 0.25)

plt.tight_layout()
plt.show()

## 6. Custom Summary Functions

Let's create custom summary functions that combine multiple statistics into tailored reports, including confidence intervals and outlier detection.

In [ ]:
def comprehensive_summary(df, group_by=None):
    """Generate a comprehensive statistical summary, optionally grouped by a variable"""
    
    # Function to generate statistics for a dataframe
    def get_stats(df_input):
        numeric_df = df_input.select_dtypes(include=[np.number])
        stats_dict = {}
        
        for col in numeric_df.columns:
            data = numeric_df[col].dropna()
            
            # Basic statistics
            stats = {
                'count': len(data),
                'missing': numeric_df[col].isna().sum(),
                'missing_pct': (numeric_df[col].isna().sum() / len(numeric_df)) * 100,
                'mean': data.mean(),
                'median': data.median(),
                'std': data.std(),
                'cv': (data.std() / data.mean()) * 100 if data.mean() != 0 else np.nan,  # Coefficient of variation
                'min': data.min(),
                'max': data.max(),
                'range': data.max() - data.min(),
                'q25': data.quantile(0.25),
                'q75': data.quantile(0.75),
                'iqr': data.quantile(0.75) - data.quantile(0.25),
                'skewness': data.skew(),
                'kurtosis': data.kurt()
            }
            
            # Calculate 95% confidence interval
            if len(data) > 1:
                std_err = stats.sem(data)
                ci_95 = std_err * stats.t.ppf((1 + 0.95) / 2, len(data) - 1)
                stats.update({
                    'ci_95_lower': stats['mean'] - ci_95,
                    'ci_95_upper': stats['mean'] + ci_95,
                    'ci_95_width': ci_95 * 2
                })
            
            # Detect outliers (outside 1.5 * IQR)
            q1, q3 = data.quantile(0.25), data.quantile(0.75)
            iqr = q3 - q1
            lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
            outliers = data[(data < lower_bound) | (data > upper_bound)]
            
            stats.update({
                'outliers_count': len(outliers),
                'outliers_pct': (len(outliers) / len(data)) * 100 if len(data) > 0 else 0
            })
            
            stats_dict[col] = stats
            
        return pd.DataFrame(stats_dict).T
    
    if group_by is None:
        return get_stats(df)
    else:
        result_dict = {}
        for group_name, group_df in df.groupby(group_by):
            result_dict[group_name] = get_stats(group_df)
        return result_dict

# Generate comprehensive summary for tips dataset
tips_summary = comprehensive_summary(tips_df)
print("Comprehensive Summary for Tips Dataset:")
display(tips_summary)

# Generate grouped summaries for tips dataset by day
tips_day_summary = comprehensive_summary(tips_df, group_by='day')
for day, summary in tips_day_summary.items():
    print(f"\nSummary for {day}:")
    display(summary)

### Customized Summary with Outlier Detection

Let's create a more focused custom summary with outlier detection and visualization.

In [ ]:
def outlier_analysis(df, column, group_by=None):
    """Perform and visualize outlier analysis for a specific column"""
    
    def detect_outliers(series):
        q1, q3 = series.quantile(0.25), series.quantile(0.75)
        iqr = q3 - q1
        lower_bound, upper_bound = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        outliers = series[(series < lower_bound) | (series > upper_bound)]
        return outliers, lower_bound, upper_bound
    
    if group_by is None:
        # Overall outlier analysis
        data = df[column].dropna()
        outliers, lower, upper = detect_outliers(data)
        
        print(f"Outlier Analysis for '{column}'")
        print(f"Number of outliers: {len(outliers)} ({len(outliers)/len(data):.2%} of data)")
        print(f"Lower bound: {lower:.2f}, Upper bound: {upper:.2f}")
        
        if not outliers.empty:
            print("Top 10 outliers:")
            display(outliers.sort_values(ascending=False).head(10))
        
        # Visualize with boxplot
        plt.figure(figsize=(10, 6))
        sns.boxplot(y=df[column])
        plt.title(f"Box Plot of {column} with Outliers", fontsize=14)
        plt.grid(True, alpha=0.3)
        
        # Highlight outliers with scatter points
        if not outliers.empty:
            outlier_indices = outliers.index
            plt.scatter(x=[0] * len(outlier_indices), y=df.loc[outlier_indices, column], 
                       color='red', s=30, label=f'Outliers ({len(outliers)})')
            plt.legend()
        
        plt.show()
        
    else:
        # Grouped outlier analysis
        fig, ax = plt.subplots(figsize=(12, 8))
        
        # Box plot by group
        sns.boxplot(x=group_by, y=column, data=df, ax=ax)
        plt.title(f"Box Plot of {column} by {group_by} with Outliers", fontsize=14)
        plt.grid(True, alpha=0.3)
        
        # Print statistics and highlight outliers for each group
        for name, group in df.groupby(group_by):
            data = group[column].dropna()
            outliers, lower, upper = detect_outliers(data)
            
            print(f"\nOutlier Analysis for '{column}' in {group_by}='{name}'")
            print(f"Number of outliers: {len(outliers)} ({len(outliers)/len(data):.2%} of data)")
            print(f"Lower bound: {lower:.2f}, Upper bound: {upper:.2f}")
            
            if not outliers.empty and len(outliers) < 20:  # Only show if not too many
                print("Outliers:")
                display(outliers.sort_values(ascending=False))
        
        plt.show()

# Perform outlier analysis for total_bill in tips dataset
outlier_analysis(tips_df, 'total_bill')

# Perform outlier analysis for total_bill grouped by day
outlier_analysis(tips_df, 'total_bill', group_by='day')

# Perform outlier analysis for normal distribution in synthetic dataset
outlier_analysis(synthetic_df_clean, 'normal')

## 7. Automated Statistical Reports

Let's demonstrate how to use libraries like pandas-profiling or sweetviz to generate comprehensive statistical reports with minimal code.

**Note**: You may need to install these libraries first:
- `pip install pandas-profiling[notebook]` or `pip install ydata-profiling[notebook]`
- `pip install sweetviz`

In [ ]:
# Try to import pandas-profiling (newer version is ydata-profiling)
try:
    try:
        from pandas_profiling import ProfileReport
        profile_module = "pandas_profiling"
    except ImportError:
        from ydata_profiling import ProfileReport
        profile_module = "ydata_profiling"
    
    # Generate a profile report for the tips dataset
    print(f"Generating profile report using {profile_module}...")
    profile = ProfileReport(tips_df, title="Tips Dataset Profiling Report", explorative=True)
    
    # Display the report in notebook
    display(profile.to_notebook_iframe())
    
    # Save the report to HTML file
    # profile.to_file("tips_profile_report.html")
    
except ImportError:
    print("Warning: pandas-profiling or ydata-profiling is not installed.")
    print("To install, run: pip install pandas-profiling[notebook] OR pip install ydata-profiling[notebook]")

In [ ]:
# Try to use sweetviz for automated EDA reporting
try:
    import sweetviz as sv
    
    # Generate a sweetviz report for the tips dataset
    print("Generating Sweetviz report...")
    sweet_report = sv.analyze(tips_df)
    
    # Display the report in notebook
    sweet_report.show_notebook()
    
    # Save the report to HTML file
    # sweet_report.show_html("tips_sweetviz_report.html")
    
except ImportError:
    print("Warning: sweetviz is not installed.")
    print("To install, run: pip install sweetviz")

## 8. Exporting Summary Statistics

Let's demonstrate how to export our summary statistics to various formats for sharing with stakeholders.

In [ ]:
# Function to export summary statistics to various formats
def export_summary_statistics(df, filename_prefix, formats=None):
    """
    Export summary statistics to various formats
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe to generate statistics for
    filename_prefix : str
        Prefix for output filenames
    formats : list
        List of formats to export (csv, excel, html, json)
    """
    if formats is None:
        formats = ['csv', 'excel', 'html', 'json']
    
    # Generate comprehensive statistics
    summary = comprehensive_summary(df)
    
    # Create a styled HTML version for better visualization
    styled_summary = summary.style.background_gradient(cmap='coolwarm', subset=['mean', 'median', 'std'])\
                                 .highlight_max(color='yellow', subset=['outliers_count', 'missing_pct'])\
                                 .set_caption("Comprehensive Statistical Summary")
    
    # Export to selected formats
    if 'csv' in formats:
        summary.to_csv(f"{filename_prefix}_summary_stats.csv")
        print(f"Exported to {filename_prefix}_summary_stats.csv")
    
    if 'excel' in formats:
        try:
            with pd.ExcelWriter(f"{filename_prefix}_summary_stats.xlsx") as writer:
                summary.to_excel(writer, sheet_name="Summary Statistics")
                df.describe().to_excel(writer, sheet_name="Basic Statistics")
                
                # Add correlation matrix if there are numeric columns
                if df.select_dtypes(include=[np.number]).shape[1] > 1:
                    df.corr().to_excel(writer, sheet_name="Correlation Matrix")
                    
            print(f"Exported to {filename_prefix}_summary_stats.xlsx")
        except Exception as e:
            print(f"Failed to export to Excel: {e}")
    
    if 'html' in formats:
        # Export both basic HTML and styled HTML
        summary.to_html(f"{filename_prefix}_summary_stats.html")
        styled_html = styled_summary.to_html()
        with open(f"{filename_prefix}_summary_stats_styled.html", "w") as f:
            f.write(styled_html)
        print(f"Exported to {filename_prefix}_summary_stats.html and {filename_prefix}_summary_stats_styled.html")
    
    if 'json' in formats:
        summary.to_json(f"{filename_prefix}_summary_stats.json", orient="index")
        print(f"Exported to {filename_prefix}_summary_stats.json")
    
    return summary

# Export tips dataset summary (comment out to avoid creating files)
# tips_exported_summary = export_summary_statistics(tips_df, "tips", formats=['csv', 'html'])
print("To export summaries, uncomment the export_summary_statistics function calls")

### Creating a PDF Report

We can also create a comprehensive PDF report using libraries like ReportLab or using Matplotlib to save figures that can be included in reports.

In [ ]:
def create_summary_figures(df, filename_prefix=None, save_figures=False):
    """Create a series of summary figures for reporting"""
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    
    # Create a multi-page figure with key visualizations
    figures = []
    
    # 1. Distribution plots for numeric variables
    for i in range(0, len(numeric_cols), 3):  # Process 3 columns per figure
        subset_cols = numeric_cols[i:i+3]
        if len(subset_cols) > 0:
            fig, axes = plt.subplots(len(subset_cols), 1, figsize=(10, 3*len(subset_cols)))
            if len(subset_cols) == 1:
                axes = [axes]
            
            for j, col in enumerate(subset_cols):
                sns.histplot(df[col].dropna(), kde=True, ax=axes[j])
                axes[j].set_title(f'Distribution of {col}')
                axes[j].set_xlabel(col)
            
            plt.tight_layout()
            figures.append(fig)
            
            if save_figures and filename_prefix:
                fig.savefig(f"{filename_prefix}_distributions_{i}.pdf", bbox_inches='tight')
    
    # 2. Correlation heatmap
    if len(numeric_cols) > 1:
        fig = plt.figure(figsize=(10, 8))
        corr_matrix = df[numeric_cols].corr()
        sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", 
                   linewidths=0.5, vmin=-1, vmax=1)
        plt.title("Correlation Matrix", fontsize=16)
        plt.tight_layout()
        figures.append(fig)
        
        if save_figures and filename_prefix:
            fig.savefig(f"{filename_prefix}_correlation.pdf", bbox_inches='tight')
    
    # 3. Box plots
    if len(numeric_cols) > 0:
        fig, ax = plt.subplots(figsize=(12, 6))
        df[numeric_cols].boxplot(ax=ax)
        plt.title("Box Plots of Numeric Variables", fontsize=16)
        plt.xticks(rotation=45)
        plt.tight_layout()
        figures.append(fig)
        
        if save_figures and filename_prefix:
            fig.savefig(f"{filename_prefix}_boxplots.pdf", bbox_inches='tight')
    
    # Display figures in notebook
    for fig in figures:
        plt.figure(fig.number)
        plt.show()
    
    return figures

# Create summary figures for tips dataset
tips_figures = create_summary_figures(tips_df, "tips", save_figures=False)

# To save the figures, set save_figures=True
# tips_figures = create_summary_figures(tips_df, "tips", save_figures=True)

## Summary

In this notebook, we've covered comprehensive techniques for calculating and reporting summary statistics:

1. **Basic descriptive statistics**: We calculated measures of central tendency, dispersion, and shape.
2. **Advanced statistical summaries**: We created confidence intervals and used groupby operations for segmented analysis.
3. **Visual statistical reporting**: We generated various visualizations to represent our data's statistical properties.
4. **Custom summary functions**: We built tailored reports with outlier detection and comprehensive statistics.
5. **Automated reporting**: We demonstrated how libraries like pandas-profiling can generate extensive reports with minimal code.
6. **Exporting capabilities**: We showed how to export statistics to various formats for sharing with stakeholders.

These techniques provide a solid foundation for any data science project, ensuring that you understand your data thoroughly before proceeding to more advanced analysis or modeling steps.